# 単語のイメージ画像を生成する


- [LCEL+structured outputの使い方](https://qiita.com/ssc-ymuramatsu/items/120e87d4a4751ab608f9)
- [サンプルコード](https://zenn.dev/asap/articles/1d5e6ec06c380f)

In [ ]:
"""Gemini APIを使用して画像を生成するスクリプト。

このスクリプトは、LangChainとGoogle Generative AI APIを利用して、
指定されたプロンプトに基づいた画像を生成し、ローカルに保存します。

主な機能:
- 指定されたテキストプロンプトから画像を生成します。
- 生成された画像をbase64からデコードし、ファイルとして保存します。
- 設定可能なパラメータ（生成枚数、出力ディレクトリなど）に対応します。
- 非同期処理による効率的な画像生成が可能です。
"""

import asyncio
import base64
import datetime
import os
from io import BytesIO
from typing import Any, Dict, List

import nest_asyncio
from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import Runnable
from langchain_google_genai import ChatGoogleGenerativeAI, Modality
from PIL import Image

nest_asyncio.apply()


def get_api_key() -> str:
    """環境変数からGoogle APIキーを取得します。

    .envファイルからAPIキーを読み込みます。
    キーが存在しない場合はValueErrorを送出します。

    Returns:
        str: Google APIキー。

    Raises:
        ValueError: APIキーが環境変数に設定されていない場合。
    """
    _ = load_dotenv(find_dotenv())
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("GOOGLE_API_KEYが環境変数に設定されていません。")
    return api_key


def base64_to_image(base64_str: str) -> Image.Image:
    """base64文字列をPillowのImageオブジェクトに変換します。

    Args:
        base64_str (str): 画像のbase64エンコード文字列。

    Returns:
        Image.Image: PillowのImageオブジェクト。
    """
    img_data = base64.b64decode(base64_str)
    return Image.open(BytesIO(img_data))


def save_image(
    image: Image.Image, filename: str = "outputs/new_generation/test.png"
) -> str:
    """画像をファイルとして保存します。

    ファイル名はタイムスタンプを元に一意に生成されます。

    Args:
        image (Image.Image): 保存するPillowのImageオブジェクト。
        output_dir (str, optional): 画像の保存先ディレクトリ。
                                    デフォルトは "outputs/new_generation"。

    Returns:
        str: 保存された画像のファイルパス。

    Raises:
        IOError: ファイルの保存に失敗した場合。
    """

    os.makedirs(os.path.dirname(filename), exist_ok=True)

    try:
        image.save(filename)
        print(f"画像を保存しました: {filename}")
        return filename
    except IOError as e:
        print(f"エラー: 画像の保存に失敗しました - {e}")
        raise


def process_generation_results(contents: List[Dict[str, str]], filename: str):
    """LLMの生成結果を処理し、画像を保存、テキストを表示します。

    Args:
        contents (List[Dict[str, str]]):
            LLMの出力結果のリスト。テキストは'str'キー、
            画像は'base64'キーを持つ辞書で構成されます。
        output_dir (str): 画像の保存先ディレクトリ。
    """
    print("=" * 12, " 生成結果 ", "=" * 12)
    for res in contents:
        if "base64" in res:
            image = base64_to_image(res["base64"])
            save_image(image, filename)
        elif "str" in res:
            print(f"出力テキスト: {res['str']}")
    print("=" * 38)


def extract_content_from_response(response: Any) -> List[Dict[str, str]]:
    """LLMのレスポンスからテキストとbase64画像を抽出します。

    Args:
        response (Any): LLMからの生のレスポンスオブジェクト。

    Returns:
        List[Dict[str, str]]: 抽出されたコンテンツのリスト。
                               例: [{'str': 'テキスト'}, {'base64': '...'}]

    Raises:
        ValueError: レスポンスの形式が不正な場合。
    """
    if not hasattr(response, "content"):
        raise ValueError("レスポンスに'content'属性が存在しません。")

    content = response.content
    if not isinstance(content, list) or not content:
        print(f'生成結果: "{content}"')
        raise ValueError("レスポンスの'content'が空またはリスト形式ではありません。")

    extracted_list = []
    for i, item in enumerate(content):
        if isinstance(item, str):
            extracted_list.append({"str": item})
        elif isinstance(item, dict):
            image_url = item.get("image_url")
            if not isinstance(image_url, dict) or "url" not in image_url:
                raise ValueError(f"content[{i}]の'image_url'の形式が不正です。")

            url = image_url["url"]
            if not isinstance(url, str) or "," not in url:
                raise ValueError(f"content[{i}]の'url'の形式が不正です。")

            base64_str = url.split(",")[-1]
            extracted_list.append({"base64": base64_str})
        else:
            raise ValueError(f"content[{i}]は想定外の型です: {type(item)}")

    return extracted_list


def create_generation_chain(api_key: str) -> Runnable:
    """画像生成用のLangChainチェーンを作成します。

    Args:
        api_key (str): Google APIキー。

    Returns:
        Runnable: 画像生成を実行するためのLangChainチェーン。
    """
    model = ChatGoogleGenerativeAI(
        model="models/gemini-2.0-flash-exp-image-generation",  # gemini-2.5-flash-image-preview
        google_api_key=api_key,
        response_modalities=[Modality.IMAGE, Modality.TEXT],
    )

    # Gemini 2.0 Flash Image Generationは現在SystemMessageをサポートしていません。
    # そのため、HumanMessageでタスクの指示を与えます。
    prompt_messages = [
        HumanMessage(
            content="""# Objective
Your task is to generate an high quality image. Generate a new image based on the content specified by the user.

----Below is the user input----
"""
        ),
        HumanMessagePromptTemplate.from_template(
            [{"type": "text", "text": "{user_input}"}]
        ),
    ]

    prompt = ChatPromptTemplate.from_messages(prompt_messages)
    return prompt | model


async def generate_image_async(
    chain: Runnable, query: str, generation_number: int
) -> List[Dict[str, str]]:
    """非同期で単一の画像を生成し、結果を抽出します。

    Args:
        chain (Runnable): LangChainの実行可能チェーン。
        query (str): 画像生成のためのテキストプロンプト。
        generation_number (int): 現在の生成回数（ログ出力用）。

    Returns:
        List[Dict[str, str]]: 抽出されたコンテンツのリスト。
    """
    print(f"画像{generation_number}の生成を開始します。")
    try:
        response = await chain.ainvoke(
            {"user_input": query},
            generation_config=dict(response_modalities=["TEXT", "IMAGE"]),
        )
        return extract_content_from_response(response)
    except ValueError as e:
        print(f"画像{generation_number}の生成中にエラーが発生しました: {e}")
        return []
    except Exception as e:
        print(f"予期せぬエラーが発生しました: {e}")
        return []


import asyncio


async def main(word: str, word_id: int):
    """スクリプトのメインエントリポイント。

    画像生成の設定を行い、非同期で画像を生成・保存します。
    """
    try:
        api_key = get_api_key()
    except ValueError as e:
        print(f"エラー: {e}")
        return

    # --- 設定 ---
    num_generations = 1  # 生成する画像の枚数 (RPM 10を考慮)
    output_directory = "outputs/generated_images/"
    output_filename = f"{output_directory}/{word_id}_{word}.png"

    # 既存ファイルがあればスキップ
    if os.path.exists(output_filename):
        print(f"ファイルが既に存在します: {output_filename}。スキップします。")
        return

    # 実際の生成
    chain = create_generation_chain(api_key)

    query = f"""Generate the image of the English word "{word}", which is used in the context of English learning.
    Make sure the image is relevant to the meaning of the word and is visually appealing.
    Provide the image without any additional text in it.
    For example, if the word is "apple", generate an image of a fresh apple.
    """
    query = f"""
Create a high-quality, realistic image that clearly illustrates the meaning of the English word "{word}" for English learners. 
The image should make it easy to understand the word's meaning visually, without any text, symbols, or captions. 
Avoid abstract or metaphorical interpretations. 
Focus on a simple, clear, and visually appealing scene directly representing the main sense of the word.
For example:
- If the word is "apple", show a realistic apple on a clean background.
- If the word is "run", show a person running outdoors.
- If the word is "happy", show a person smiling naturally.
Use natural lighting, balanced composition, and educational clarity.
Place the word itself at the bottom right corner in small, unobtrusive text.
The aspect ratio should be 1:1.
    """

    tasks = [generate_image_async(chain, query, i + 1) for i in range(num_generations)]
    results = await asyncio.gather(*tasks)

    for extracted_content in results:
        print(extracted_content)
        if extracted_content:
            process_generation_results(extracted_content, output_filename)


# 同時実行数を制限するラッパー関数
async def run_with_semaphore(semaphore, name, port):
    async with semaphore:
        return await main(name, port)


# if __name__ == "__main__":
#     asyncio.run(main())

In [ ]:
# 一つのみの実行テスト
asyncio.run(main("symposium", 6001))

In [ ]:
# メイン関数
# 内部で引数リストをループして非同期実行
async def run_all(params: List):
    semaphore = asyncio.Semaphore(3)  # 同時実行数を10に制限

    # すべてのタスクを作成
    tasks = [run_with_semaphore(semaphore, name, port) for name, port in params]

    # gatherで全タスクを実行
    results = await asyncio.gather(*tasks)
    print("All done:", results)

In [ ]:
# 実行したい引数セットをリストに
params = [
    ("symposium", 6001),
    ("broaden", 6002),
    ("ecology", 6003),
    ("circulate", 6004),
    ("corpse", 6005),
    # ... 必要なだけ追加
]

asyncio.run(run_all(params))

画像1の生成を開始します。
画像1の生成を開始します。
画像1の生成を開始します。
画像1の生成を開始します。
画像1の生成を開始します。
[{'str': 'I will generate a realistic image depicting a balanced natural environment to illustrate the meaning of "ecology". The scene will feature a small pond with fish, surrounded by various plants, insects, and a small animal, showcasing the interconnectedness of living organisms and their environment. The lighting will be natural, suggesting a sunny day, and the composition will be balanced to clearly show these interactions. The word "ecology" will be placed in small, unobtrusive text at the bottom right corner.'}, {'base64': '...'}]
============  生成結果  ============
出力テキスト: I will generate a realistic image depicting a balanced natural environment to illustrate the meaning of "ecology". The scene will feature a small pond with fish, surrounded by various plants, insects, and a small animal, showcasing the interconnectedness of living organisms and their environment. The lighting will be natural, suggesting a su

In [ ]:
import os
import time
from logging import DEBUG, INFO, StreamHandler, getLogger
from typing import List

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI
from pydantic import BaseModel, Field


def load_input_csv(input_file: str):
    # 入力CSVの読み込み(重複なしの想定)
    try:
        df = pd.read_csv(
            input_file, dtype={"word_id": int, "word": str}, na_filter=False
        )
    except Exception as e:
        print(f"CSVファイルの読み込みエラー: {e}")
        raise ValueError(f"CSVファイルの読み込みエラー: {e}")
    return df

In [ ]:
# 使用例
if __name__ == "__main__":
    # input_file = "../generate_examples/lv6.csv"  # 入力ファイル名
    input_file = "../word_data/lv6.csv"
    # input_file = "../word_data/lv6.csv"
    # output_file = "../explanation_data/lv6_explanation.csv"  # 出力ファイル名

    df = load_input_csv(input_file)

    # 各単語を処理
    params = [(row["word"], row["word_id"]) for index, row in df.iterrows()]

In [ ]:
asyncio.run(run_all(params))

ファイルが既に存在します: outputs/generated_images//5001_trivial.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5002_misunderstanding.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5003_hostility.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5004_accumulate.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5005_tourism.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5006_boom.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5007_unsteady.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5008_jolly.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5009_brighten.png。スキップします。
ファイルが既に存在します: outputs/generated_images//5010_depressed.png。スキップします。
画像1の生成を開始します。
画像1の生成を開始します。
画像1の生成を開始します。
[{'base64': '...'}]
============  生成結果  ============
画像を保存しました: outputs/generated_images//5012_unconscious.png
画像1の生成を開始します。
